In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset as SurpriseDataset, Reader
from surprise import accuracy
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

# 1M 데이터 로드
ratings = pd.read_csv('../data/raw/ratings.dat', sep='::',
                      names=['userId','movieId','rating','timestamp'],
                      engine='python')

movies = pd.read_csv('../data/raw/movies.dat', sep='::',
                     names=['movieId','title','genres'],
                     engine='python', encoding='latin-1')

# 시간 기준 분리
ratings = ratings.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(ratings) * 0.8)
train_df = ratings.iloc[:split_idx].copy()
test_df  = ratings.iloc[split_idx:].copy()

# 인덱스 변환
user_ids  = train_df['userId'].unique()
movie_ids = train_df['movieId'].unique()
user2idx  = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}

train_df['user_idx']  = train_df['userId'].map(user2idx)
train_df['movie_idx'] = train_df['movieId'].map(movie2idx)
test_df['user_idx']   = test_df['userId'].map(user2idx).fillna(0).astype(int)
test_df['movie_idx']  = test_df['movieId'].map(movie2idx).fillna(0).astype(int)

n_users  = len(user2idx)
n_movies = len(movie2idx)
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Train: {len(train_df):,}개  |  Test: {len(test_df):,}개")
print(f"유저: {n_users:,}  |  영화: {n_movies:,}  |  장치: {device}")

Train: 800,167개  |  Test: 200,042개
유저: 5,400  |  영화: 3,662  |  장치: cpu


In [2]:
def compute_temporal_weights(df, lam=0.1):
    t_max  = df['timestamp'].max()
    t_min  = df['timestamp'].min()
    t_half = (t_max - t_min) / 2
    weights = np.exp(-lam * (t_max - df['timestamp']) / (t_half + 1))
    return weights.values

train_weights = compute_temporal_weights(train_df, lam=0.1)

class RatingDataset(Dataset):
    def __init__(self, df, weights=None, timestamps=None):
        self.users      = torch.LongTensor(df['user_idx'].values)
        self.movies     = torch.LongTensor(df['movie_idx'].values)
        self.ratings    = torch.FloatTensor(df['rating'].values)
        self.weights    = torch.FloatTensor(
            weights if weights is not None else np.ones(len(df))
        )
        # Time Embedding용 정규화 타임스탬프 (0~1)
        if timestamps is not None:
            t_min = timestamps.min()
            t_max = timestamps.max()
            self.timestamps = torch.FloatTensor(
                ((timestamps - t_min) / (t_max - t_min + 1)).values
            )
        else:
            self.timestamps = torch.zeros(len(df))

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (self.users[idx], self.movies[idx],
                self.ratings[idx], self.weights[idx],
                self.timestamps[idx])

train_dataset = RatingDataset(train_df, train_weights, train_df['timestamp'])
test_dataset  = RatingDataset(test_df,  None,          test_df['timestamp'])

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False)

print(f"Train 배치: {len(train_loader)}  |  Test 배치: {len(test_loader)}")
print("Dataset 준비 완료")

Train 배치: 1563  |  Test 배치: 391
Dataset 준비 완료


C:\Users\Ryu\AppData\Local\Temp\ipykernel_35596\2986104852.py:12: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self.users      = torch.LongTensor(df['user_idx'].values)


In [3]:
class TemporalNCF_Final(nn.Module):
    """
    약점 1 해결: Batch Normalization 추가
    약점 2 해결: Time Embedding 추가 (타임스탬프를 모델이 직접 학습)
    """
    def __init__(self, n_users, n_movies, embed_dim=32, time_dim=8, layers=[64, 32, 16]):
        super().__init__()

        # 유저/영화 임베딩
        self.user_embedding  = nn.Embedding(n_users,  embed_dim)
        self.movie_embedding = nn.Embedding(n_movies, embed_dim)

        # Time Embedding — 타임스탬프 값을 벡터로 변환
        self.time_encoder = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )

        # MLP with Batch Normalization
        mlp_layers = []
        input_dim  = embed_dim * 2 + time_dim
        for out_dim in layers:
            mlp_layers.append(nn.Linear(input_dim, out_dim))
            mlp_layers.append(nn.BatchNorm1d(out_dim))  # 약점 1 해결
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = out_dim
        mlp_layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, user_idx, movie_idx, weights=None, timestamps=None):
        u = self.user_embedding(user_idx)
        m = self.movie_embedding(movie_idx)

        # 시간 감쇠 가중치를 임베딩에 적용
        if weights is not None:
            w = weights.unsqueeze(1)
            u = u * w
            m = m * w

        # Time Embedding — 약점 2 해결
        if timestamps is not None:
            t = timestamps.unsqueeze(1)
            t_embed = self.time_encoder(t)
        else:
            t_embed = torch.zeros(user_idx.size(0), 8).to(user_idx.device)

        x   = torch.cat([u, m, t_embed], dim=1)
        out = self.mlp(x)
        out = torch.sigmoid(out) * 4 + 1
        return out.squeeze()

# 학습 함수
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    criterion  = nn.MSELoss(reduction='none')
    for users, movies, ratings, weights, timestamps in loader:
        users     = users.to(device)
        movies    = movies.to(device)
        ratings   = ratings.to(device)
        weights   = weights.to(device)
        timestamps = timestamps.to(device)

        optimizer.zero_grad()
        preds = model(users, movies, weights, timestamps)
        loss  = (criterion(preds, ratings) * weights).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    preds_list, targets_list = [], []
    with torch.no_grad():
        for users, movies, ratings, weights, timestamps in loader:
            users      = users.to(device)
            movies     = movies.to(device)
            timestamps = timestamps.to(device)
            preds = model(users, movies, timestamps=timestamps)
            preds_list.extend(preds.cpu().numpy())
            targets_list.extend(ratings.numpy())
    preds_arr   = np.array(preds_list)
    targets_arr = np.array(targets_list)
    rmse = np.sqrt(np.mean((preds_arr - targets_arr) ** 2))
    mae  = np.mean(np.abs(preds_arr - targets_arr))
    return rmse, mae

# 학습 실행
model_final = TemporalNCF_Final(n_users, n_movies,
                                 embed_dim=32, time_dim=8,
                                 layers=[64, 32, 16])
model_final = model_final.to(device)
optimizer   = torch.optim.Adam(model_final.parameters(), lr=0.001, weight_decay=1e-5)

print("TemporalNCF Final 학습 시작\n")
best_rmse  = float('inf')
best_epoch = 0
no_improve = 0
patience   = 5

for epoch in range(1, 31):
    train_loss = train_epoch(model_final, train_loader, optimizer, device)
    rmse, mae  = evaluate(model_final, test_loader, device)

    if rmse < best_rmse:
        best_rmse  = rmse
        best_epoch = epoch
        no_improve = 0
        torch.save(model_final.state_dict(), '../results/best_ncf_final.pt')
    else:
        no_improve += 1

    if epoch % 5 == 0 or no_improve == 0:
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} {'★' if no_improve==0 else ''}")

    if no_improve >= patience:
        print(f"\nEarly Stopping (patience={patience})")
        break

print(f"\n최적 Epoch: {best_epoch}  |  RMSE: {best_rmse:.4f}")
print(f"\n이전 TemporalNCF:  0.9704")
print(f"TemporalNCF Final: {best_rmse:.4f}")
print(f"개선폭: {0.9704 - best_rmse:.4f}")

TemporalNCF Final 학습 시작

Epoch  1 | Loss: 1.0807 | RMSE: 1.0145 | MAE: 0.7939 ★
Epoch  2 | Loss: 0.8544 | RMSE: 0.9686 | MAE: 0.7686 ★
Epoch  5 | Loss: 0.7632 | RMSE: 1.0222 | MAE: 0.8059 

Early Stopping (patience=5)

최적 Epoch: 2  |  RMSE: 0.9686

이전 TemporalNCF:  0.9704
TemporalNCF Final: 0.9686
개선폭: 0.0018


In [4]:
# 1M 데이터로 TemporalSVD 재측정
reader = Reader(rating_scale=(1, 5))

def apply_temporal_weight(df, lam=0.1):
    t_max  = df['timestamp'].max()
    t_min  = df['timestamp'].min()
    t_half = (t_max - t_min) / 2
    weights = np.exp(-lam * (t_max - df['timestamp']) / (t_half + 1))
    df = df.copy()
    df['weight'] = weights
    return df

weighted_df = apply_temporal_weight(train_df, lam=0.1)
global_mean = weighted_df['rating'].mean()
weighted_df['rating_adjusted'] = (
    weighted_df['weight'] * weighted_df['rating'] +
    (1 - weighted_df['weight']) * global_mean
)

svd_data  = SurpriseDataset.load_from_df(
    weighted_df[['userId','movieId','rating_adjusted']]
    .rename(columns={'rating_adjusted':'rating'}), reader
)
svd_trainset = svd_data.build_full_trainset()
model_A = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
model_A.fit(svd_trainset)

testset  = [(r['userId'], r['movieId'], r['rating']) for _, r in test_df.iterrows()]
preds_A  = model_A.test(testset)

rmse_A = accuracy.rmse(preds_A, verbose=False)
mae_A  = accuracy.mae(preds_A,  verbose=False)
print(f"모델 A (TemporalSVD) — RMSE: {rmse_A:.4f}  MAE: {mae_A:.4f}")

모델 A (TemporalSVD) — RMSE: 0.9400  MAE: 0.7435


In [6]:
# 1M movies.dat 장르 파싱
movies['genre_list'] = movies['genres'].str.split('|')
movies['genre_text'] = movies['genre_list'].apply(lambda x: ' '.join(x))

tfidf      = TfidfVectorizer()
tfidf_mat  = tfidf.fit_transform(movies['genre_text'])
cosine_sim = cosine_similarity(tfidf_mat, tfidf_mat)

movie_ids_list = movies['movieId'].tolist()
id_to_idx      = {mid: idx for idx, mid in enumerate(movie_ids_list)}

# 유저별 평균 장르 벡터로 빠르게 예측
print("CBF 빠른 버전 예측 중...")

# 유저별 평균 장르 프로필 미리 계산
user_profiles = {}
for uid, group in train_df.groupby('userId'):
    idxs = [id_to_idx[mid] for mid in group['movieId'] if mid in id_to_idx]
    if len(idxs) == 0:
        continue
    ratings_arr = group[group['movieId'].isin(id_to_idx)]['rating'].values
    vectors     = tfidf_mat[idxs].toarray()
    # 평점 가중 평균으로 유저 프로필 생성
    weights     = ratings_arr / ratings_arr.sum()
    user_profiles[uid] = np.dot(weights, vectors)

global_mean_cbf = train_df['rating'].mean()

cbf_preds_raw = []
for _, row in test_df.iterrows():
    uid = row['userId']
    mid = row['movieId']

    if uid not in user_profiles or mid not in id_to_idx:
        pred = global_mean_cbf
    else:
        movie_vec    = tfidf_mat[id_to_idx[mid]].toarray().flatten()
        user_vec     = user_profiles[uid]
        norm_u = np.linalg.norm(user_vec)
        norm_m = np.linalg.norm(movie_vec)
        if norm_u == 0 or norm_m == 0:
            pred = global_mean_cbf
        else:
            sim  = np.dot(user_vec, movie_vec) / (norm_u * norm_m)
            pred = global_mean_cbf + sim * 2
    pred = np.clip(pred, 1, 5)
    cbf_preds_raw.append((uid, mid, row['rating'], pred, None))

from surprise.prediction_algorithms.predictions import Prediction
preds_B = [Prediction(uid, iid, r_ui, est, None)
           for uid, iid, r_ui, est, _ in cbf_preds_raw]

rmse_B = accuracy.rmse(preds_B, verbose=False)
mae_B  = accuracy.mae(preds_B,  verbose=False)
print(f"모델 B (CBF) — RMSE: {rmse_B:.4f}  MAE: {mae_B:.4f}")

CBF 빠른 버전 예측 중...
모델 B (CBF) — RMSE: 1.3088  MAE: 1.0455


In [7]:
# 모델 C (TemporalNCF Final) predictions 생성
model_final.eval()
ncf_preds_raw = []
with torch.no_grad():
    for users, movies_t, ratings, weights, timestamps in test_loader:
        users      = users.to(device)
        movies_t   = movies_t.to(device)
        timestamps = timestamps.to(device)
        preds = model_final(users, movies_t, timestamps=timestamps)
        ncf_preds_raw.extend(preds.cpu().numpy())

preds_C = [Prediction(row['userId'], row['movieId'], row['rating'],
                       float(np.clip(p, 1, 5)), None)
           for (_, row), p in zip(test_df.iterrows(), ncf_preds_raw)]

rmse_C = accuracy.rmse(preds_C, verbose=False)
mae_C  = accuracy.mae(preds_C,  verbose=False)
print(f"모델 C (TemporalNCF) — RMSE: {rmse_C:.4f}  MAE: {mae_C:.4f}\n")

# 예측값 딕셔너리 변환
pred_dict_A = {(p.uid, p.iid): p.est for p in preds_A}
pred_dict_B = {(p.uid, p.iid): p.est for p in preds_B}
pred_dict_C = {(p.uid, p.iid): p.est for p in preds_C}

def hybrid_predict(test_df, weights_dict, pred_dicts, global_mean):
    """
    weights_dict: {'A': 0.5, 'B': 0.3, 'C': 0.2} 형태
    pred_dicts:   {'A': dict, 'B': dict, 'C': dict} 형태
    """
    preds = []
    for _, row in test_df.iterrows():
        key = (row['userId'], row['movieId'])
        est = sum(w * pred_dicts[m].get(key, global_mean)
                  for m, w in weights_dict.items())
        est = np.clip(est, 1, 5)
        preds.append(Prediction(key[0], key[1], row['rating'], est, None))
    return preds

gm = train_df['rating'].mean()
all_preds = {'A': pred_dict_A, 'B': pred_dict_B, 'C': pred_dict_C}

# 7개 조합 정의
combos = {
    'A+B':     {'A': 0.7, 'B': 0.3},
    'A+C':     {'A': 0.3, 'C': 0.7},
    'B+C':     {'B': 0.3, 'C': 0.7},
    'A+B+C':   {'A': 0.3, 'B': 0.2, 'C': 0.5},
}

print("하이브리드 조합 계산 중...\n")
results = [
    {'모델': 'A (TemporalSVD)',    'RMSE': round(rmse_A, 4), 'MAE': round(mae_A, 4)},
    {'모델': 'B (CBF)',            'RMSE': round(rmse_B, 4), 'MAE': round(mae_B, 4)},
    {'모델': 'C (TemporalNCF)',    'RMSE': round(rmse_C, 4), 'MAE': round(mae_C, 4)},
]

for name, weights in combos.items():
    preds = hybrid_predict(test_df, weights, all_preds, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    mae   = accuracy.mae(preds,  verbose=False)
    results.append({'모델': name, 'RMSE': round(rmse,4), 'MAE': round(mae,4)})
    print(f"{name:<10} → RMSE: {rmse:.4f}  MAE: {mae:.4f}")

results_df = pd.DataFrame(results).sort_values('RMSE')
print("\n=== 전체 모델 비교 (RMSE 오름차순) ===\n")
print(results_df.to_string(index=False))

results_df.to_csv('../results/final_all_models.csv', index=False)
print("\n저장 완료 → results/final_all_models.csv")

모델 C (TemporalNCF) — RMSE: 1.1028  MAE: 0.8666

하이브리드 조합 계산 중...

A+B        → RMSE: 0.9859  MAE: 0.7798
A+C        → RMSE: 1.0148  MAE: 0.8035
B+C        → RMSE: 1.0410  MAE: 0.8288
A+B+C      → RMSE: 0.9869  MAE: 0.7858

=== 전체 모델 비교 (RMSE 오름차순) ===

             모델   RMSE    MAE
A (TemporalSVD) 0.9400 0.7435
            A+B 0.9859 0.7798
          A+B+C 0.9869 0.7858
            A+C 1.0148 0.8035
            B+C 1.0410 0.8288
C (TemporalNCF) 1.1028 0.8666
        B (CBF) 1.3088 1.0455

저장 완료 → results/final_all_models.csv


In [8]:
print("Grid Search 시작 — 최적 가중치 탐색\n")

best_results = {}

# A+B 최적화
best_ab = {'rmse': float('inf'), 'alpha': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha, 1), 'B': round(1-alpha, 1)}
    preds = hybrid_predict(test_df, w, all_preds, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ab['rmse']:
        best_ab = {'rmse': rmse, 'alpha': alpha}
best_results['A+B'] = best_ab
print(f"A+B  최적 → A:{best_ab['alpha']:.1f} B:{1-best_ab['alpha']:.1f}  RMSE: {best_ab['rmse']:.4f}")

# A+C 최적화
best_ac = {'rmse': float('inf'), 'alpha': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha, 1), 'C': round(1-alpha, 1)}
    preds = hybrid_predict(test_df, w, all_preds, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ac['rmse']:
        best_ac = {'rmse': rmse, 'alpha': alpha}
best_results['A+C'] = best_ac
print(f"A+C  최적 → A:{best_ac['alpha']:.1f} C:{1-best_ac['alpha']:.1f}  RMSE: {best_ac['rmse']:.4f}")

# B+C 최적화
best_bc = {'rmse': float('inf'), 'alpha': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'B': round(alpha, 1), 'C': round(1-alpha, 1)}
    preds = hybrid_predict(test_df, w, all_preds, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_bc['rmse']:
        best_bc = {'rmse': rmse, 'alpha': alpha}
best_results['B+C'] = best_bc
print(f"B+C  최적 → B:{best_bc['alpha']:.1f} C:{1-best_bc['alpha']:.1f}  RMSE: {best_bc['rmse']:.4f}")

# A+B+C 최적화
best_abc = {'rmse': float('inf'), 'w': None}
for a in np.arange(0.1, 0.9, 0.1):
    for b in np.arange(0.1, 0.9-a, 0.1):
        c = round(1 - a - b, 1)
        if c <= 0:
            continue
        w = {'A': round(a,1), 'B': round(b,1), 'C': round(c,1)}
        preds = hybrid_predict(test_df, w, all_preds, gm)
        rmse  = accuracy.rmse(preds, verbose=False)
        if rmse < best_abc['rmse']:
            best_abc = {'rmse': rmse, 'w': w}
best_results['A+B+C'] = best_abc
print(f"A+B+C최적 → {best_abc['w']}  RMSE: {best_abc['rmse']:.4f}")

# 최종 비교표
print("\n=== 최종 비교표 (Grid Search 적용) ===\n")
final_rows = [
    {'모델': 'A (TemporalSVD)',  'RMSE': round(rmse_A, 4), 'MAE': round(mae_A, 4), '가중치': '-'},
    {'모델': 'B (CBF)',          'RMSE': round(rmse_B, 4), 'MAE': round(mae_B, 4), '가중치': '-'},
    {'모델': 'C (TemporalNCF)',  'RMSE': round(rmse_C, 4), 'MAE': round(mae_C, 4), '가중치': '-'},
    {'모델': 'A+B',  'RMSE': round(best_ab['rmse'],4),  'MAE': '-',
     '가중치': f"A:{best_ab['alpha']:.1f} B:{1-best_ab['alpha']:.1f}"},
    {'모델': 'A+C',  'RMSE': round(best_ac['rmse'],4),  'MAE': '-',
     '가중치': f"A:{best_ac['alpha']:.1f} C:{1-best_ac['alpha']:.1f}"},
    {'모델': 'B+C',  'RMSE': round(best_bc['rmse'],4),  'MAE': '-',
     '가중치': f"B:{best_bc['alpha']:.1f} C:{1-best_bc['alpha']:.1f}"},
    {'모델': 'A+B+C','RMSE': round(best_abc['rmse'],4), 'MAE': '-',
     '가중치': str(best_abc['w'])},
]

final_df = pd.DataFrame(final_rows).sort_values('RMSE')
print(final_df.to_string(index=False))

final_df.to_csv('../results/final_grid_search.csv', index=False)
print("\n저장 완료 → results/final_grid_search.csv")

Grid Search 시작 — 최적 가중치 탐색

A+B  최적 → A:0.9 B:0.1  RMSE: 0.9475
A+C  최적 → A:0.9 C:0.1  RMSE: 0.9380
B+C  최적 → B:0.3 C:0.7  RMSE: 1.0410
A+B+C최적 → {'A': 0.7, 'B': 0.1, 'C': 0.2}  RMSE: 0.9461

=== 최종 비교표 (Grid Search 적용) ===

             모델   RMSE     MAE                            가중치
            A+C 0.9380       -                    A:0.9 C:0.1
A (TemporalSVD) 0.9400  0.7435                              -
          A+B+C 0.9461       - {'A': 0.7, 'B': 0.1, 'C': 0.2}
            A+B 0.9475       -                    A:0.9 B:0.1
            B+C 1.0410       -                    B:0.3 C:0.7
C (TemporalNCF) 1.1028  0.8666                              -
        B (CBF) 1.3088  1.0455                              -

저장 완료 → results/final_grid_search.csv


In [10]:
class PeriodicTimeEncoder(nn.Module):
    """
    타임스탬프를 주기적 특징으로 분해
    - 연도, 월, 요일, 시간대를 sin/cos로 인코딩
    """
    def __init__(self, out_dim=16):
        super().__init__()
        self.fc = nn.Linear(8, out_dim)

    def forward(self, timestamps):
        # timestamps: 0~1로 정규화된 값
        # 다양한 주기로 sin/cos 변환
        t = timestamps.unsqueeze(1)
        features = torch.cat([
            torch.sin(2 * np.pi * t),        # 1년 주기
            torch.cos(2 * np.pi * t),
            torch.sin(4 * np.pi * t),        # 반년 주기
            torch.cos(4 * np.pi * t),
            torch.sin(12 * np.pi * t),       # 월 주기
            torch.cos(12 * np.pi * t),
            torch.sin(52 * np.pi * t),       # 주 주기
            torch.cos(52 * np.pi * t),
        ], dim=1)
        return self.fc(features)


class NeuMF_Temporal(nn.Module):
    """
    NeuMF 구조 + Periodic Time Feature
    GMF(선형 상호작용) + MLP(비선형 상호작용) 결합
    """
    def __init__(self, n_users, n_movies,
                 gmf_dim=32, mlp_dim=32,
                 time_dim=16, layers=[64, 32, 16]):
        super().__init__()

        # GMF 임베딩 (선형 상호작용용)
        self.gmf_user  = nn.Embedding(n_users,  gmf_dim)
        self.gmf_movie = nn.Embedding(n_movies, gmf_dim)

        # MLP 임베딩 (비선형 상호작용용)
        self.mlp_user  = nn.Embedding(n_users,  mlp_dim)
        self.mlp_movie = nn.Embedding(n_movies, mlp_dim)

        # Periodic Time Encoder
        self.time_encoder = PeriodicTimeEncoder(out_dim=time_dim)

        # MLP 레이어 with Batch Normalization
        mlp_layers = []
        input_dim  = mlp_dim * 2 + time_dim
        for out_dim in layers:
            mlp_layers.append(nn.Linear(input_dim, out_dim))
            mlp_layers.append(nn.BatchNorm1d(out_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = out_dim
        self.mlp = nn.Sequential(*mlp_layers)

        # 최종 출력 레이어 (GMF + MLP 결합)
        self.output = nn.Linear(gmf_dim + layers[-1], 1)

        # 임베딩 초기화
        nn.init.normal_(self.gmf_user.weight,  std=0.01)
        nn.init.normal_(self.gmf_movie.weight, std=0.01)
        nn.init.normal_(self.mlp_user.weight,  std=0.01)
        nn.init.normal_(self.mlp_movie.weight, std=0.01)

    def forward(self, user_idx, movie_idx, weights=None, timestamps=None):
        # GMF 경로
        gmf_u = self.gmf_user(user_idx)
        gmf_m = self.gmf_movie(movie_idx)

        # 시간 감쇠 가중치 적용
        if weights is not None:
            w = weights.unsqueeze(1)
            gmf_u = gmf_u * w
            gmf_m = gmf_m * w

        gmf_out = gmf_u * gmf_m  # element-wise 곱

        # MLP 경로
        mlp_u = self.mlp_user(user_idx)
        mlp_m = self.mlp_movie(movie_idx)

        if weights is not None:
            mlp_u = mlp_u * w
            mlp_m = mlp_m * w

        # Periodic Time Feature
        if timestamps is not None:
            t_feat = self.time_encoder(timestamps)
        else:
            t_feat = torch.zeros(user_idx.size(0), 16).to(user_idx.device)

        mlp_input = torch.cat([mlp_u, mlp_m, t_feat], dim=1)
        mlp_out   = self.mlp(mlp_input)

        # GMF + MLP 결합
        combined = torch.cat([gmf_out, mlp_out], dim=1)
        out = self.output(combined)
        out = torch.sigmoid(out) * 4 + 1
        return out.squeeze()


# 학습 실행
model_neumf = NeuMF_Temporal(n_users, n_movies,
                              gmf_dim=32, mlp_dim=32,
                              time_dim=16, layers=[64, 32, 16])
model_neumf = model_neumf.to(device)

total_params = sum(p.numel() for p in model_neumf.parameters())
print(f"NeuMF_Temporal 파라미터 수: {total_params:,}")

optimizer = torch.optim.Adam(model_neumf.parameters(),
                              lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print("\nNeuMF Temporal 학습 시작\n")
best_rmse  = float('inf')
best_epoch = 0
no_improve = 0
patience   = 5

for epoch in range(1, 31):
    train_loss = train_epoch(model_neumf, train_loader, optimizer, device)
    rmse, mae  = evaluate(model_neumf, test_loader, device)
    scheduler.step(rmse)

    if rmse < best_rmse:
        best_rmse  = rmse
        best_epoch = epoch
        no_improve = 0
        torch.save(model_neumf.state_dict(), '../results/best_neumf_temporal.pt')
    else:
        no_improve += 1

    if epoch % 3 == 0 or no_improve == 0:
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | "
              f"RMSE: {rmse:.4f} | MAE: {mae:.4f} "
              f"{'★ best' if no_improve == 0 else ''}")

    if no_improve >= patience:
        print(f"\nEarly Stopping (patience={patience})")
        break

print(f"\n최적 Epoch: {best_epoch}  |  RMSE: {best_rmse:.4f}")
print(f"\n=== NCF 개선 비교 ===")
print(f"기존 TemporalNCF:     1.0418  (100k)")
print(f"TemporalNCF Final:   0.9686  (1M + BN + TimeEmbed)")
print(f"NeuMF_Temporal:      {best_rmse:.4f}  (1M + BN + Periodic + NeuMF)")
print(f"\n목표: TemporalSVD(0.9400) 이하")

NeuMF_Temporal 파라미터 수: 588,177

NeuMF Temporal 학습 시작

Epoch  1 | Loss: 0.8391 | RMSE: 1.0101 | MAE: 0.7998 ★ best
Epoch  3 | Loss: 0.6938 | RMSE: 1.1627 | MAE: 0.9114 
Epoch  6 | Loss: 0.6168 | RMSE: 1.1792 | MAE: 0.9206 

Early Stopping (patience=5)

최적 Epoch: 1  |  RMSE: 1.0101

=== NCF 개선 비교 ===
기존 TemporalNCF:     1.0418  (100k)
TemporalNCF Final:   0.9686  (1M + BN + TimeEmbed)
NeuMF_Temporal:      1.0101  (1M + BN + Periodic + NeuMF)

목표: TemporalSVD(0.9400) 이하


In [11]:
class NeuMF_Temporal_v2(nn.Module):
    def __init__(self, n_users, n_movies,
                 gmf_dim=16, mlp_dim=16, time_dim=8):
        super().__init__()

        # GMF 임베딩
        self.gmf_user  = nn.Embedding(n_users,  gmf_dim)
        self.gmf_movie = nn.Embedding(n_movies, gmf_dim)

        # MLP 임베딩
        self.mlp_user  = nn.Embedding(n_users,  mlp_dim)
        self.mlp_movie = nn.Embedding(n_movies, mlp_dim)

        # Periodic Time Encoder (경량화)
        self.time_encoder = nn.Sequential(
            nn.Linear(4, time_dim),
            nn.ReLU()
        )

        # MLP (레이어 줄임)
        self.mlp = nn.Sequential(
            nn.Linear(mlp_dim * 2 + time_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
        )

        # 출력
        self.output = nn.Linear(gmf_dim + 16, 1)

        nn.init.normal_(self.gmf_user.weight,  std=0.01)
        nn.init.normal_(self.gmf_movie.weight, std=0.01)
        nn.init.normal_(self.mlp_user.weight,  std=0.01)
        nn.init.normal_(self.mlp_movie.weight, std=0.01)

    def forward(self, user_idx, movie_idx, weights=None, timestamps=None):
        # GMF
        gmf_u = self.gmf_user(user_idx)
        gmf_m = self.gmf_movie(movie_idx)
        if weights is not None:
            w = weights.unsqueeze(1)
            gmf_u = gmf_u * w
            gmf_m = gmf_m * w
        gmf_out = gmf_u * gmf_m

        # MLP
        mlp_u = self.mlp_user(user_idx)
        mlp_m = self.mlp_movie(movie_idx)
        if weights is not None:
            mlp_u = mlp_u * w
            mlp_m = mlp_m * w

        # 간단한 Periodic Feature (sin/cos 2개)
        if timestamps is not None:
            t = timestamps.unsqueeze(1)
            t_feat = torch.cat([
                torch.sin(2 * np.pi * t),
                torch.cos(2 * np.pi * t),
                torch.sin(4 * np.pi * t),
                torch.cos(4 * np.pi * t),
            ], dim=1)
            t_feat = self.time_encoder(t_feat)
        else:
            t_feat = torch.zeros(user_idx.size(0), 8).to(user_idx.device)

        mlp_out = self.mlp(torch.cat([mlp_u, mlp_m, t_feat], dim=1))

        out = self.output(torch.cat([gmf_out, mlp_out], dim=1))
        out = torch.sigmoid(out) * 4 + 1
        return out.squeeze()

model_v2 = NeuMF_Temporal_v2(n_users, n_movies)
model_v2  = model_v2.to(device)

total_params = sum(p.numel() for p in model_v2.parameters())
print(f"파라미터 수: {total_params:,}  (이전: 588,177)")

optimizer_v2 = torch.optim.Adam(model_v2.parameters(),
                                  lr=0.001, weight_decay=1e-4)

print("\nNeuMF_Temporal v2 학습 시작\n")
best_rmse2  = float('inf')
best_epoch2 = 0
no_improve2 = 0

for epoch in range(1, 31):
    train_loss = train_epoch(model_v2, train_loader, optimizer_v2, device)
    rmse, mae  = evaluate(model_v2, test_loader, device)

    if rmse < best_rmse2:
        best_rmse2  = rmse
        best_epoch2 = epoch
        no_improve2 = 0
        torch.save(model_v2.state_dict(), '../results/best_neumf_v2.pt')
    else:
        no_improve2 += 1

    if epoch % 3 == 0 or no_improve2 == 0:
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | "
              f"RMSE: {rmse:.4f} | {'★' if no_improve2==0 else ''}")

    if no_improve2 >= 5:
        print(f"\nEarly Stopping")
        break

print(f"\n최적 Epoch: {best_epoch2}  |  RMSE: {best_rmse2:.4f}")
print(f"\n=== 전체 NCF 계보 ===")
print(f"TemporalNCF (100k):     1.0418")
print(f"TemporalNCF Final (1M): 0.9686")
print(f"NeuMF v1 (1M):          1.0101")
print(f"NeuMF v2 (1M):          {best_rmse2:.4f}")
print(f"\n목표: TemporalSVD(0.9400) 이하")

파라미터 수: 291,961  (이전: 588,177)

NeuMF_Temporal v2 학습 시작

Epoch  1 | Loss: 0.8122 | RMSE: 1.0539 | ★
Epoch  2 | Loss: 0.7502 | RMSE: 1.0410 | ★
Epoch  3 | Loss: 0.7319 | RMSE: 1.0729 | 
Epoch  6 | Loss: 0.6960 | RMSE: 1.1287 | 

Early Stopping

최적 Epoch: 2  |  RMSE: 1.0410

=== 전체 NCF 계보 ===
TemporalNCF (100k):     1.0418
TemporalNCF Final (1M): 0.9686
NeuMF v1 (1M):          1.0101
NeuMF v2 (1M):          1.0410

목표: TemporalSVD(0.9400) 이하


In [12]:
# 영화 제목에서 연도 추출
import re

def extract_year(title):
    match = re.search(r'\((\d{4})\)', title)
    if match:
        year = int(match.group(1))
        if 1900 <= year <= 2023:
            return year
    return None

movies['year'] = movies['title'].apply(extract_year)
movies['year'] = movies['year'].fillna(movies['year'].median())

print(f"연도 추출 성공: {movies['year'].notna().sum()}개")
print(f"연도 범위: {movies['year'].min():.0f} ~ {movies['year'].max():.0f}")

# 영화별 평균 평점 및 평점 수 계산 (인기도)
movie_stats = train_df.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

movies = movies.merge(movie_stats, on='movieId', how='left')
movies['avg_rating']   = movies['avg_rating'].fillna(movies['avg_rating'].median())
movies['rating_count'] = movies['rating_count'].fillna(0)

# 연도 정규화 (0~1)
year_min = movies['year'].min()
year_max = movies['year'].max()
movies['year_norm'] = (movies['year'] - year_min) / (year_max - year_min + 1)

# 인기도 정규화 (log scale)
movies['popularity_norm'] = np.log1p(movies['rating_count'])
movies['popularity_norm'] = movies['popularity_norm'] / movies['popularity_norm'].max()

print(f"\n평균 평점 범위: {movies['avg_rating'].min():.2f} ~ {movies['avg_rating'].max():.2f}")
print(f"인기도 정규화 완료")

연도 추출 성공: 3883개
연도 범위: 1919 ~ 2000

평균 평점 범위: 1.00 ~ 5.00
인기도 정규화 완료


In [13]:
# 장르 + 연도 + 인기도를 결합한 향상된 특성 벡터 구축
all_genres = sorted(set(
    g for genres in movies['genre_list'] for g in genres
))

for genre in all_genres:
    if genre not in movies.columns:
        movies[genre] = movies['genre_list'].apply(
            lambda x: 1 if genre in x else 0
        )

# TF-IDF 장르 벡터
tfidf_v2     = TfidfVectorizer()
tfidf_mat_v2 = tfidf_v2.fit_transform(movies['genre_text'])
tfidf_dense  = tfidf_mat_v2.toarray()

# 연도 + 인기도 가중치 조정 (장르 80%, 연도 15%, 인기도 5%)
year_vec       = movies['year_norm'].values.reshape(-1, 1) * 0.15
popularity_vec = movies['popularity_norm'].values.reshape(-1, 1) * 0.05
genre_vec      = tfidf_dense * 0.80

# 결합 특성 벡터
combined_features = np.hstack([genre_vec, year_vec, popularity_vec])

# 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim_v2  = cosine_similarity(combined_features, combined_features)

movie_ids_list_v2 = movies['movieId'].tolist()
id_to_idx_v2      = {mid: idx for idx, mid in enumerate(movie_ids_list_v2)}

print(f"결합 특성 벡터 크기: {combined_features.shape}")
print(f"코사인 유사도 행렬: {cosine_sim_v2.shape}")

# 유저 프로필 사전 계산 (개선된 버전)
print("\n유저 프로필 계산 중...")
user_profiles_v2 = {}
for uid, group in train_df.groupby('userId'):
    valid = group[group['movieId'].isin(id_to_idx_v2)]
    if len(valid) == 0:
        continue
    idxs        = [id_to_idx_v2[mid] for mid in valid['movieId']]
    ratings_arr = valid['rating'].values
    # 평점 가중 평균
    w           = ratings_arr / ratings_arr.sum()
    user_profiles_v2[uid] = np.dot(w, combined_features[idxs])

global_mean_v2 = train_df['rating'].mean()

# 예측
print("CBF v2 예측 중...")
cbf_v2_preds_raw = []
for _, row in test_df.iterrows():
    uid = row['userId']
    mid = row['movieId']

    if uid not in user_profiles_v2 or mid not in id_to_idx_v2:
        pred = global_mean_v2
    else:
        movie_vec = combined_features[id_to_idx_v2[mid]]
        user_vec  = user_profiles_v2[uid]
        norm_u = np.linalg.norm(user_vec)
        norm_m = np.linalg.norm(movie_vec)
        if norm_u == 0 or norm_m == 0:
            pred = global_mean_v2
        else:
            sim  = np.dot(user_vec, movie_vec) / (norm_u * norm_m)
            # 유사도 + 영화 평균 평점 반영
            movie_avg = movies[movies['movieId'] == mid]['avg_rating'].values
            movie_avg = movie_avg[0] if len(movie_avg) > 0 else global_mean_v2
            pred = 0.7 * (global_mean_v2 + sim * 2) + 0.3 * movie_avg
    pred = np.clip(pred, 1, 5)
    cbf_v2_preds_raw.append((uid, mid, row['rating'], pred, None))

preds_B_v2 = [Prediction(uid, iid, r_ui, est, None)
              for uid, iid, r_ui, est, _ in cbf_v2_preds_raw]

rmse_B_v2 = accuracy.rmse(preds_B_v2, verbose=False)
mae_B_v2  = accuracy.mae(preds_B_v2,  verbose=False)

print(f"\n기존 CBF RMSE:    1.3088")
print(f"개선된 CBF RMSE:  {rmse_B_v2:.4f}")
print(f"개선폭: {1.3088 - rmse_B_v2:.4f}")

결합 특성 벡터 크기: (3883, 22)
코사인 유사도 행렬: (3883, 3883)

유저 프로필 계산 중...
CBF v2 예측 중...

기존 CBF RMSE:    1.3088
개선된 CBF RMSE:  1.1938
개선폭: 0.1150


In [14]:
# 개선된 B로 pred_dict 업데이트
pred_dict_B_v2 = {(p.uid, p.iid): p.est for p in preds_B_v2}

# C 모델 — TemporalNCF Final (0.9686) predictions 다시 생성
model_final.load_state_dict(
    torch.load('../results/best_ncf_final.pt', map_location=device)
)
model_final.eval()
ncf_preds_raw = []
with torch.no_grad():
    for users, movies_t, ratings, weights, timestamps in test_loader:
        users      = users.to(device)
        movies_t   = movies_t.to(device)
        timestamps = timestamps.to(device)
        preds = model_final(users, movies_t, timestamps=timestamps)
        ncf_preds_raw.extend(preds.cpu().numpy())

preds_C_final = [Prediction(row['userId'], row['movieId'], row['rating'],
                             float(np.clip(p, 1, 5)), None)
                 for (_, row), p in zip(test_df.iterrows(), ncf_preds_raw)]

rmse_C_final = accuracy.rmse(preds_C_final, verbose=False)
pred_dict_C_final = {(p.uid, p.iid): p.est for p in preds_C_final}

all_preds_v2 = {
    'A': pred_dict_A,
    'B': pred_dict_B_v2,
    'C': pred_dict_C_final
}
gm = train_df['rating'].mean()

print("최종 Grid Search 시작\n")

# A+B
best_ab = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha,1), 'B': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_v2, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ab['rmse']:
        best_ab = {'rmse': rmse, 'w': w}
print(f"A+B   → {best_ab['w']}  RMSE: {best_ab['rmse']:.4f}")

# A+C
best_ac = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha,1), 'C': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_v2, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ac['rmse']:
        best_ac = {'rmse': rmse, 'w': w}
print(f"A+C   → {best_ac['w']}  RMSE: {best_ac['rmse']:.4f}")

# B+C
best_bc = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'B': round(alpha,1), 'C': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_v2, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_bc['rmse']:
        best_bc = {'rmse': rmse, 'w': w}
print(f"B+C   → {best_bc['w']}  RMSE: {best_bc['rmse']:.4f}")

# A+B+C
best_abc = {'rmse': float('inf'), 'w': None}
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 0.9-a, 0.05):
        c = round(1 - a - b, 2)
        if c <= 0:
            continue
        w = {'A': round(a,2), 'B': round(b,2), 'C': round(c,2)}
        preds = hybrid_predict(test_df, w, all_preds_v2, gm)
        rmse  = accuracy.rmse(preds, verbose=False)
        if rmse < best_abc['rmse']:
            best_abc = {'rmse': rmse, 'w': w}
print(f"A+B+C → {best_abc['w']}  RMSE: {best_abc['rmse']:.4f}")

# 최종 비교표
print("\n=== 최종 전체 모델 비교표 ===\n")
final_rows = [
    {'모델': 'A  (TemporalSVD)',  'RMSE': round(rmse_A, 4),        '가중치': '-'},
    {'모델': 'B  (CBF v2)',       'RMSE': round(rmse_B_v2, 4),     '가중치': '-'},
    {'모델': 'C  (TemporalNCF)',  'RMSE': round(rmse_C_final, 4),  '가중치': '-'},
    {'모델': 'A+B',  'RMSE': round(best_ab['rmse'],4),  '가중치': str(best_ab['w'])},
    {'모델': 'A+C',  'RMSE': round(best_ac['rmse'],4),  '가중치': str(best_ac['w'])},
    {'모델': 'B+C',  'RMSE': round(best_bc['rmse'],4),  '가중치': str(best_bc['w'])},
    {'모델': 'A+B+C','RMSE': round(best_abc['rmse'],4), '가중치': str(best_abc['w'])},
]

final_df = pd.DataFrame(final_rows).sort_values('RMSE').reset_index(drop=True)
final_df.index += 1
print(final_df.to_string())

final_df.to_csv('../results/final_comparison_v2.csv', index=False)
print("\n저장 완료 → results/final_comparison_v2.csv")

최종 Grid Search 시작

A+B   → {'A': 0.9, 'B': 0.1}  RMSE: 0.9450
A+C   → {'A': 0.8, 'C': 0.2}  RMSE: 0.9384
B+C   → {'B': 0.1, 'C': 0.9}  RMSE: 0.9703
A+B+C → {'A': 0.75, 'B': 0.1, 'C': 0.15}  RMSE: 0.9441

=== 최종 전체 모델 비교표 ===

                 모델    RMSE                               가중치
1               A+C  0.9384              {'A': 0.8, 'C': 0.2}
2  A  (TemporalSVD)  0.9400                                 -
3             A+B+C  0.9441  {'A': 0.75, 'B': 0.1, 'C': 0.15}
4               A+B  0.9450              {'A': 0.9, 'B': 0.1}
5  C  (TemporalNCF)  0.9686                                 -
6               B+C  0.9703              {'B': 0.1, 'C': 0.9}
7       B  (CBF v2)  1.1938                                 -

저장 완료 → results/final_comparison_v2.csv


In [15]:
def precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls, ndcgs = {}, {}, {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k  = user_ratings[:k]
        n_rel  = sum(1 for (_, r) in user_ratings if r >= threshold)
        n_hit  = sum(1 for (_, r) in top_k if r >= threshold)

        precisions[uid] = n_hit / k
        recalls[uid]    = n_hit / n_rel if n_rel > 0 else 0

        dcg  = sum((2**(r >= threshold) - 1) / np.log2(i + 2)
                   for i, (_, r) in enumerate(top_k))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(n_rel, k)))
        ndcgs[uid] = dcg / idcg if idcg > 0 else 0

    return (sum(precisions.values()) / len(precisions),
            sum(recalls.values())    / len(recalls),
            sum(ndcgs.values())      / len(ndcgs))

def compute_avg_ild(predictions, movies_df, genre_cols, k=10):
    user_est = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        user_est[uid].append((est, iid))

    ild_scores = []
    for uid, items in user_est.items():
        items.sort(key=lambda x: x[0], reverse=True)
        top_k_ids = [int(iid) for _, iid in items[:k]]
        valid_ids = [i for i in top_k_ids if i in movies_df.index]
        if len(valid_ids) < 2:
            continue
        vectors = movies_df.loc[valid_ids, genre_cols].values.astype(float)
        n = len(vectors)
        total, count = 0, 0
        for i in range(n):
            for j in range(i+1, n):
                ni = np.linalg.norm(vectors[i])
                nj = np.linalg.norm(vectors[j])
                if ni > 0 and nj > 0:
                    total += 1 - np.dot(vectors[i], vectors[j]) / (ni * nj)
                    count += 1
        if count > 0:
            ild_scores.append(total / count)
    return np.mean(ild_scores) if ild_scores else 0

# ILD용 장르 행렬 준비
movies_indexed = movies.set_index('movieId')
genre_cols_1m  = [g for g in all_genres if g in movies_indexed.columns]

print("함수 정의 완료")
print(f"장르 컬럼 수: {len(genre_cols_1m)}")

함수 정의 완료
장르 컬럼 수: 18


In [16]:
# 하이브리드 predictions 생성
def get_hybrid_preds(test_df, weights, pred_dicts, gm):
    preds = []
    for _, row in test_df.iterrows():
        key = (row['userId'], row['movieId'])
        est = sum(w * pred_dicts[m].get(key, gm)
                  for m, w in weights.items())
        est = np.clip(est, 1, 5)
        preds.append(Prediction(key[0], key[1], row['rating'], est, None))
    return preds

preds_AB  = get_hybrid_preds(test_df, best_ab['w'],  all_preds_v2, gm)
preds_AC  = get_hybrid_preds(test_df, best_ac['w'],  all_preds_v2, gm)
preds_BC  = get_hybrid_preds(test_df, best_bc['w'],  all_preds_v2, gm)
preds_ABC = get_hybrid_preds(test_df, best_abc['w'], all_preds_v2, gm)

# 전체 모델 리스트
model_preds = [
    ('A (TemporalSVD)',  preds_A),
    ('B (CBF v2)',       preds_B_v2),
    ('C (TemporalNCF)',  preds_C_final),
    ('A+B',             preds_AB),
    ('A+C',             preds_AC),
    ('B+C',             preds_BC),
    ('A+B+C',           preds_ABC),
]

print("전체 지표 측정 중... (ILD가 오래 걸려요)\n")
all_results = []

for name, preds in model_preds:
    rmse = accuracy.rmse(preds, verbose=False)
    mae  = accuracy.mae(preds,  verbose=False)
    p, r, n = precision_recall_ndcg_at_k(preds, k=10, threshold=3.5)
    ild  = compute_avg_ild(preds, movies_indexed, genre_cols_1m, k=10)

    all_results.append({
        '모델':         name,
        'RMSE':         round(rmse, 4),
        'MAE':          round(mae,  4),
        'Precision@10': round(p,    4),
        'Recall@10':    round(r,    4),
        'NDCG@10':      round(n,    4),
        'ILD':          round(ild,  4),
    })
    print(f"{name:<18} RMSE:{rmse:.4f} P@10:{p:.4f} NDCG:{n:.4f} ILD:{ild:.4f}")

final_all_df = pd.DataFrame(all_results).sort_values('RMSE').reset_index(drop=True)
final_all_df.index += 1

print("\n=== 최종 완성 비교표 ===\n")
print(final_all_df.to_string())

final_all_df.to_csv('../results/final_complete_metrics.csv', index=False)
print("\n저장 완료 → results/final_complete_metrics.csv")

전체 지표 측정 중... (ILD가 오래 걸려요)

A (TemporalSVD)    RMSE:0.9400 P@10:0.7473 NDCG:0.8429 ILD:0.6794
B (CBF v2)         RMSE:1.1938 P@10:0.6238 NDCG:0.7036 ILD:0.5277
C (TemporalNCF)    RMSE:0.9686 P@10:0.7348 NDCG:0.8274 ILD:0.6862
A+B                RMSE:0.9450 P@10:0.7477 NDCG:0.8437 ILD:0.6698
A+C                RMSE:0.9384 P@10:0.7490 NDCG:0.8444 ILD:0.6801
B+C                RMSE:0.9703 P@10:0.7357 NDCG:0.8275 ILD:0.6755
A+B+C              RMSE:0.9441 P@10:0.7494 NDCG:0.8449 ILD:0.6695

=== 최종 완성 비교표 ===

                모델    RMSE     MAE  Precision@10  Recall@10  NDCG@10     ILD
1              A+C  0.9384  0.7430        0.7490     0.3538   0.8444  0.6801
2  A (TemporalSVD)  0.9400  0.7435        0.7473     0.3528   0.8429  0.6794
3            A+B+C  0.9441  0.7473        0.7494     0.3539   0.8449  0.6695
4              A+B  0.9450  0.7474        0.7477     0.3534   0.8437  0.6698
5  C (TemporalNCF)  0.9686  0.7686        0.7348     0.3503   0.8274  0.6862
6              B+C  0.9703 

In [17]:
from surprise.model_selection import cross_validate

print("TemporalSVD 하이퍼파라미터 튜닝 시작\n")

# 탐색할 파라미터 조합
param_grid = [
    {'n_factors': 50,  'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02},
    {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02},
    {'n_factors': 150, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02},
    {'n_factors': 200, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02},
    {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.02},
    {'n_factors': 150, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.01},
    {'n_factors': 150, 'n_epochs': 20, 'lr_all': 0.010, 'reg_all': 0.02},
    {'n_factors': 200, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.02},
]

tuning_results = []
best_rmse_tune = float('inf')
best_params    = None

for params in param_grid:
    # 시간 감쇠 적용
    weighted_df = apply_temporal_weight(train_df, lam=0.1).copy()
    global_mean = weighted_df['rating'].mean()
    weighted_df['rating_adjusted'] = (
        weighted_df['weight'] * weighted_df['rating'] +
        (1 - weighted_df['weight']) * global_mean
    )
    svd_data = SurpriseDataset.load_from_df(
        weighted_df[['userId','movieId','rating_adjusted']]
        .rename(columns={'rating_adjusted':'rating'}), reader
    )
    trainset = svd_data.build_full_trainset()

    model = SVD(
        n_factors=params['n_factors'],
        n_epochs=params['n_epochs'],
        lr_all=params['lr_all'],
        reg_all=params['reg_all'],
        random_state=42
    )
    model.fit(trainset)
    preds = model.test(testset)
    rmse  = accuracy.rmse(preds, verbose=False)
    mae   = accuracy.mae(preds,  verbose=False)

    tuning_results.append({
        'n_factors': params['n_factors'],
        'n_epochs':  params['n_epochs'],
        'lr_all':    params['lr_all'],
        'reg_all':   params['reg_all'],
        'RMSE':      round(rmse, 4),
        'MAE':       round(mae,  4),
    })

    if rmse < best_rmse_tune:
        best_rmse_tune = rmse
        best_params    = params
        best_model_A   = model
        best_preds_A   = preds

    print(f"n_factors={params['n_factors']:3d} n_epochs={params['n_epochs']} "
          f"lr={params['lr_all']} reg={params['reg_all']} "
          f"→ RMSE: {rmse:.4f}")

tuning_df = pd.DataFrame(tuning_results).sort_values('RMSE')
print(f"\n최적 파라미터: {best_params}")
print(f"최적 RMSE:    {best_rmse_tune:.4f}")
print(f"기존 RMSE:    0.9400")
print(f"개선폭:       {0.9400 - best_rmse_tune:.4f}")

tuning_df.to_csv('../results/svd_tuning_results.csv', index=False)
print("\n저장 완료 → results/svd_tuning_results.csv")

TemporalSVD 하이퍼파라미터 튜닝 시작

n_factors= 50 n_epochs=20 lr=0.005 reg=0.02 → RMSE: 0.9386
n_factors=100 n_epochs=20 lr=0.005 reg=0.02 → RMSE: 0.9400
n_factors=150 n_epochs=20 lr=0.005 reg=0.02 → RMSE: 0.9423
n_factors=200 n_epochs=20 lr=0.005 reg=0.02 → RMSE: 0.9428
n_factors=150 n_epochs=30 lr=0.005 reg=0.02 → RMSE: 0.9462
n_factors=150 n_epochs=20 lr=0.005 reg=0.01 → RMSE: 0.9504
n_factors=150 n_epochs=20 lr=0.01 reg=0.02 → RMSE: 0.9504
n_factors=200 n_epochs=30 lr=0.005 reg=0.02 → RMSE: 0.9463

최적 파라미터: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
최적 RMSE:    0.9386
기존 RMSE:    0.9400
개선폭:       0.0014

저장 완료 → results/svd_tuning_results.csv


In [18]:
print("n_factors 세밀 탐색 (10~60 구간)\n")

fine_results = []
best_rmse_fine = float('inf')
best_factor    = None

for n_factors in [10, 20, 30, 40, 50, 60, 70, 80]:
    weighted_df = apply_temporal_weight(train_df, lam=0.1).copy()
    global_mean = weighted_df['rating'].mean()
    weighted_df['rating_adjusted'] = (
        weighted_df['weight'] * weighted_df['rating'] +
        (1 - weighted_df['weight']) * global_mean
    )
    svd_data = SurpriseDataset.load_from_df(
        weighted_df[['userId','movieId','rating_adjusted']]
        .rename(columns={'rating_adjusted':'rating'}), reader
    )
    trainset = svd_data.build_full_trainset()

    model = SVD(n_factors=n_factors, n_epochs=20,
                lr_all=0.005, reg_all=0.02, random_state=42)
    model.fit(trainset)
    preds = model.test(testset)
    rmse  = accuracy.rmse(preds, verbose=False)

    fine_results.append({'n_factors': n_factors, 'RMSE': round(rmse, 4)})

    if rmse < best_rmse_fine:
        best_rmse_fine = rmse
        best_factor    = n_factors
        best_model_A   = model
        best_preds_A   = preds

    print(f"n_factors={n_factors:3d} → RMSE: {rmse:.4f}")

print(f"\n최적 n_factors: {best_factor}  (RMSE: {best_rmse_fine:.4f})")
print(f"기존 RMSE:       0.9400")
print(f"개선폭:          {0.9400 - best_rmse_fine:.4f}")

n_factors 세밀 탐색 (10~60 구간)

n_factors= 10 → RMSE: 0.9391
n_factors= 20 → RMSE: 0.9382
n_factors= 30 → RMSE: 0.9382
n_factors= 40 → RMSE: 0.9382
n_factors= 50 → RMSE: 0.9386
n_factors= 60 → RMSE: 0.9380
n_factors= 70 → RMSE: 0.9390
n_factors= 80 → RMSE: 0.9405

최적 n_factors: 60  (RMSE: 0.9380)
기존 RMSE:       0.9400
개선폭:          0.0020


In [19]:
print("최종 A 모델 (n_factors=60) 전체 지표 측정 중...\n")

# 최종 A predictions으로 업데이트
pred_dict_A_final = {(p.uid, p.iid): p.est for p in best_preds_A}
all_preds_final = {
    'A': pred_dict_A_final,
    'B': pred_dict_B_v2,
    'C': pred_dict_C_final
}

# Grid Search 재실행
best_ab2 = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha,1), 'B': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_final, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ab2['rmse']:
        best_ab2 = {'rmse': rmse, 'w': w}

best_ac2 = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'A': round(alpha,1), 'C': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_final, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_ac2['rmse']:
        best_ac2 = {'rmse': rmse, 'w': w}

best_bc2 = {'rmse': float('inf'), 'w': None}
for alpha in np.arange(0.1, 1.0, 0.1):
    w = {'B': round(alpha,1), 'C': round(1-alpha,1)}
    preds = hybrid_predict(test_df, w, all_preds_final, gm)
    rmse  = accuracy.rmse(preds, verbose=False)
    if rmse < best_bc2['rmse']:
        best_bc2 = {'rmse': rmse, 'w': w}

best_abc2 = {'rmse': float('inf'), 'w': None}
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 0.9-a, 0.05):
        c = round(1 - a - b, 2)
        if c <= 0:
            continue
        w = {'A': round(a,2), 'B': round(b,2), 'C': round(c,2)}
        preds = hybrid_predict(test_df, w, all_preds_final, gm)
        rmse  = accuracy.rmse(preds, verbose=False)
        if rmse < best_abc2['rmse']:
            best_abc2 = {'rmse': rmse, 'w': w}

print(f"A+B   최적: {best_ab2['w']}  RMSE: {best_ab2['rmse']:.4f}")
print(f"A+C   최적: {best_ac2['w']}  RMSE: {best_ac2['rmse']:.4f}")
print(f"B+C   최적: {best_bc2['w']}  RMSE: {best_bc2['rmse']:.4f}")
print(f"A+B+C 최적: {best_abc2['w']}  RMSE: {best_abc2['rmse']:.4f}")

# 전체 지표 측정
preds_AB2  = get_hybrid_preds(test_df, best_ab2['w'],  all_preds_final, gm)
preds_AC2  = get_hybrid_preds(test_df, best_ac2['w'],  all_preds_final, gm)
preds_BC2  = get_hybrid_preds(test_df, best_bc2['w'],  all_preds_final, gm)
preds_ABC2 = get_hybrid_preds(test_df, best_abc2['w'], all_preds_final, gm)

model_preds_final = [
    ('A (TemporalSVD)',  best_preds_A),
    ('B (CBF v2)',       preds_B_v2),
    ('C (TemporalNCF)',  preds_C_final),
    ('A+B',             preds_AB2),
    ('A+C',             preds_AC2),
    ('B+C',             preds_BC2),
    ('A+B+C',           preds_ABC2),
]

print("\n전체 지표 측정 중...\n")
final_results = []
for name, preds in model_preds_final:
    rmse = accuracy.rmse(preds, verbose=False)
    mae  = accuracy.mae(preds,  verbose=False)
    p, r, n = precision_recall_ndcg_at_k(preds, k=10, threshold=3.5)
    ild  = compute_avg_ild(preds, movies_indexed, genre_cols_1m, k=10)

    final_results.append({
        '모델':         name,
        'RMSE':         round(rmse, 4),
        'MAE':          round(mae,  4),
        'Precision@10': round(p,    4),
        'Recall@10':    round(r,    4),
        'NDCG@10':      round(n,    4),
        'ILD':          round(ild,  4),
    })
    print(f"{name:<18} RMSE:{rmse:.4f} P@10:{p:.4f} NDCG:{n:.4f} ILD:{ild:.4f}")

final_df = pd.DataFrame(final_results).sort_values('RMSE').reset_index(drop=True)
final_df.index += 1

print("\n=== 최종 완성 비교표 (n_factors=60 적용) ===\n")
print(final_df.to_string())

final_df.to_csv('../results/final_complete_v2.csv', index=False)
print("\n저장 완료 → results/final_complete_v2.csv")

최종 A 모델 (n_factors=60) 전체 지표 측정 중...

A+B   최적: {'A': 0.9, 'B': 0.1}  RMSE: 0.9432
A+C   최적: {'A': 0.8, 'C': 0.2}  RMSE: 0.9370
B+C   최적: {'B': 0.1, 'C': 0.9}  RMSE: 0.9703
A+B+C 최적: {'A': 0.75, 'B': 0.1, 'C': 0.15}  RMSE: 0.9427

전체 지표 측정 중...

A (TemporalSVD)    RMSE:0.9380 P@10:0.7495 NDCG:0.8438 ILD:0.6750
B (CBF v2)         RMSE:1.1938 P@10:0.6238 NDCG:0.7036 ILD:0.5277
C (TemporalNCF)    RMSE:0.9686 P@10:0.7348 NDCG:0.8274 ILD:0.6862
A+B                RMSE:0.9432 P@10:0.7500 NDCG:0.8437 ILD:0.6650
A+C                RMSE:0.9370 P@10:0.7482 NDCG:0.8444 ILD:0.6771
B+C                RMSE:0.9703 P@10:0.7357 NDCG:0.8275 ILD:0.6755
A+B+C              RMSE:0.9427 P@10:0.7486 NDCG:0.8448 ILD:0.6676

=== 최종 완성 비교표 (n_factors=60 적용) ===

                모델    RMSE     MAE  Precision@10  Recall@10  NDCG@10     ILD
1              A+C  0.9370  0.7413        0.7482     0.3531   0.8444  0.6771
2  A (TemporalSVD)  0.9380  0.7411        0.7495     0.3538   0.8438  0.6750
3            A+B+C  0.9

In [20]:
class TimeSVDpp:
    """
    TimeSVD++ 구현
    - 유저 편향이 시간에 따라 선형으로 변함
    - 영화 편향이 기간(bin)별로 다름
    - 유저 잠재 요인이 시간에 따라 변함
    """
    def __init__(self, n_factors=60, n_epochs=20, lr=0.005,
                 reg=0.02, n_bins=30, beta=0.4, random_state=42):
        self.n_factors   = n_factors
        self.n_epochs    = n_epochs
        self.lr          = lr
        self.reg         = reg
        self.n_bins      = n_bins    # 영화 편향 기간 수
        self.beta        = beta      # dev(t) 지수
        self.random_state = random_state

    def _get_bin(self, timestamp):
        """타임스탬프를 n_bins 구간 중 하나로 변환"""
        t_range = self.t_max - self.t_min
        bin_idx = int((timestamp - self.t_min) / t_range * self.n_bins)
        return min(bin_idx, self.n_bins - 1)

    def _dev(self, t, t_u_mean):
        """유저 평균 시점 대비 편차"""
        diff = t - t_u_mean
        return np.sign(diff) * (abs(diff) ** self.beta)

    def fit(self, train_df):
        np.random.seed(self.random_state)

        self.t_min = train_df['timestamp'].min()
        self.t_max = train_df['timestamp'].max()
        t_range    = self.t_max - self.t_min

        # 유저/영화 인덱스
        users  = train_df['userId'].unique()
        movies = train_df['movieId'].unique()
        self.user2idx  = {u: i for i, u in enumerate(users)}
        self.movie2idx = {m: i for i, m in enumerate(movies)}

        n_users  = len(users)
        n_movies = len(movies)

        # 파라미터 초기화
        self.mu     = train_df['rating'].mean()
        self.b_u    = np.zeros(n_users)
        self.b_i    = np.zeros(n_movies)
        self.alpha_u = np.zeros(n_users)          # 유저 편향 시간 변화율
        self.b_i_t  = np.zeros((n_movies, self.n_bins))  # 영화 기간별 편향
        self.p_u    = np.random.normal(0, 0.01, (n_users,  self.n_factors))
        self.q_i    = np.random.normal(0, 0.01, (n_movies, self.n_factors))
        self.alpha_p = np.zeros((n_users, self.n_factors))  # 유저 요인 시간 변화율

        # 유저별 평균 평점 시점 계산
        self.t_u_mean = train_df.groupby('userId')['timestamp'].mean()

        print(f"TimeSVD++ 학습 시작")
        print(f"파라미터: n_factors={self.n_factors}, n_bins={self.n_bins}, beta={self.beta}\n")

        for epoch in range(self.n_epochs):
            total_loss = 0
            shuffled = train_df.sample(frac=1, random_state=epoch)

            for _, row in shuffled.iterrows():
                u = self.user2idx.get(row['userId'])
                i = self.movie2idx.get(row['movieId'])
                if u is None or i is None:
                    continue

                t   = row['timestamp']
                r   = row['rating']
                t_u = self.t_u_mean.get(row['userId'], self.mu)
                dev = self._dev(t, t_u)
                bin_idx = self._get_bin(t)

                # 예측
                b_u_t = self.b_u[u] + self.alpha_u[u] * dev
                b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]
                p_u_t = self.p_u[u] + self.alpha_p[u] * dev

                r_hat = self.mu + b_u_t + b_i_t + np.dot(p_u_t, self.q_i[i])
                r_hat = np.clip(r_hat, 1, 5)
                err   = r - r_hat
                total_loss += err ** 2

                # SGD 업데이트
                self.b_u[u]          += self.lr * (err - self.reg * self.b_u[u])
                self.alpha_u[u]      += self.lr * (err * dev - self.reg * self.alpha_u[u])
                self.b_i[i]          += self.lr * (err - self.reg * self.b_i[i])
                self.b_i_t[i,bin_idx] += self.lr * (err - self.reg * self.b_i_t[i,bin_idx])
                self.p_u[u]          += self.lr * (err * self.q_i[i] - self.reg * self.p_u[u])
                self.alpha_p[u]      += self.lr * (err * self.q_i[i] * dev - self.reg * self.alpha_p[u])
                self.q_i[i]          += self.lr * (err * p_u_t - self.reg * self.q_i[i])

            rmse = np.sqrt(total_loss / len(shuffled))
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1:2d} | Train RMSE: {rmse:.4f}")

        return self

    def predict(self, user_id, movie_id, timestamp):
        u = self.user2idx.get(user_id)
        i = self.movie2idx.get(movie_id)

        if u is None or i is None:
            return self.mu

        t_u  = self.t_u_mean.get(user_id, self.mu)
        dev  = self._dev(timestamp, t_u)
        bin_idx = self._get_bin(timestamp)

        b_u_t = self.b_u[u] + self.alpha_u[u] * dev
        b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]
        p_u_t = self.p_u[u] + self.alpha_p[u] * dev

        r_hat = self.mu + b_u_t + b_i_t + np.dot(p_u_t, self.q_i[i])
        return float(np.clip(r_hat, 1, 5))

    def test(self, test_df):
        preds = []
        for _, row in test_df.iterrows():
            est = self.predict(row['userId'], row['movieId'], row['timestamp'])
            preds.append(Prediction(row['userId'], row['movieId'],
                                    row['rating'], est, None))
        return preds

# 학습 실행
time_svdpp = TimeSVDpp(n_factors=60, n_epochs=20, lr=0.005,
                        reg=0.02, n_bins=30, beta=0.4)
time_svdpp.fit(train_df)

print("\nTest 평가 중...")
preds_timesvd = time_svdpp.test(test_df)
rmse_timesvd  = accuracy.rmse(preds_timesvd, verbose=False)
mae_timesvd   = accuracy.mae(preds_timesvd,  verbose=False)

print(f"\n=== TimeSVD++ 결과 ===")
print(f"RMSE: {rmse_timesvd:.4f}")
print(f"MAE:  {mae_timesvd:.4f}")
print(f"\nTemporalSVD (A): 0.9380")
print(f"TimeSVD++:       {rmse_timesvd:.4f}")
print(f"개선폭:           {0.9380 - rmse_timesvd:.4f}")

TimeSVD++ 학습 시작
파라미터: n_factors=60, n_bins=30, beta=0.4

Epoch  5 | Train RMSE: nan
Epoch 10 | Train RMSE: nan
Epoch 15 | Train RMSE: nan
Epoch 20 | Train RMSE: nan

Test 평가 중...

=== TimeSVD++ 결과 ===
RMSE: nan
MAE:  nan

TemporalSVD (A): 0.9380
TimeSVD++:       nan
개선폭:           nan


In [21]:
class TimeSVDpp_v2:
    def __init__(self, n_factors=60, n_epochs=20, lr=0.001,
                 reg=0.02, n_bins=30, beta=0.4, random_state=42):
        self.n_factors    = n_factors
        self.n_epochs     = n_epochs
        self.lr           = lr
        self.reg          = reg
        self.n_bins       = n_bins
        self.beta         = beta
        self.random_state = random_state

    def _get_bin(self, t_norm):
        bin_idx = int(t_norm * self.n_bins)
        return min(bin_idx, self.n_bins - 1)

    def _dev(self, t_norm, t_u_mean_norm):
        diff = t_norm - t_u_mean_norm
        return np.sign(diff) * (abs(diff) ** self.beta)

    def fit(self, train_df):
        np.random.seed(self.random_state)

        # 타임스탬프 0~1 정규화
        self.t_min = train_df['timestamp'].min()
        self.t_max = train_df['timestamp'].max()
        t_range    = self.t_max - self.t_min + 1

        train_df = train_df.copy()
        train_df['t_norm'] = (train_df['timestamp'] - self.t_min) / t_range

        users  = train_df['userId'].unique()
        movies = train_df['movieId'].unique()
        self.user2idx  = {u: i for i, u in enumerate(users)}
        self.movie2idx = {m: i for i, m in enumerate(movies)}

        n_users  = len(users)
        n_movies = len(movies)

        self.mu      = train_df['rating'].mean()
        self.b_u     = np.zeros(n_users)
        self.b_i     = np.zeros(n_movies)
        self.alpha_u = np.zeros(n_users)
        self.b_i_t   = np.zeros((n_movies, self.n_bins))
        self.p_u     = np.random.normal(0, 0.01, (n_users,  self.n_factors))
        self.q_i     = np.random.normal(0, 0.01, (n_movies, self.n_factors))
        self.alpha_p = np.zeros((n_users, self.n_factors))

        # 유저별 평균 평점 시점 (정규화)
        self.t_u_mean = train_df.groupby('userId')['t_norm'].mean()

        print(f"TimeSVD++ v2 학습 시작")
        print(f"lr={self.lr}, n_factors={self.n_factors}, n_bins={self.n_bins}\n")

        for epoch in range(self.n_epochs):
            total_loss = 0
            shuffled   = train_df.sample(frac=1, random_state=epoch)

            for _, row in shuffled.iterrows():
                u = self.user2idx.get(row['userId'])
                i = self.movie2idx.get(row['movieId'])
                if u is None or i is None:
                    continue

                t_norm  = row['t_norm']
                r       = row['rating']
                t_u     = self.t_u_mean.get(row['userId'], 0.5)
                dev     = self._dev(t_norm, t_u)
                bin_idx = self._get_bin(t_norm)

                b_u_t = self.b_u[u] + self.alpha_u[u] * dev
                b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]
                p_u_t = self.p_u[u] + self.alpha_p[u] * dev

                r_hat = self.mu + b_u_t + b_i_t + np.dot(p_u_t, self.q_i[i])
                r_hat = np.clip(r_hat, 1, 5)
                err   = r - r_hat
                total_loss += err ** 2

                # gradient clipping
                err_clipped = np.clip(err, -5, 5)

                self.b_u[u]           += self.lr * (err_clipped - self.reg * self.b_u[u])
                self.alpha_u[u]       += self.lr * (err_clipped * dev - self.reg * self.alpha_u[u])
                self.b_i[i]           += self.lr * (err_clipped - self.reg * self.b_i[i])
                self.b_i_t[i,bin_idx] += self.lr * (err_clipped - self.reg * self.b_i_t[i,bin_idx])
                self.p_u[u]           += self.lr * (err_clipped * self.q_i[i] - self.reg * self.p_u[u])
                self.alpha_p[u]       += self.lr * (err_clipped * self.q_i[i] * dev - self.reg * self.alpha_p[u])
                self.q_i[i]           += self.lr * (err_clipped * p_u_t - self.reg * self.q_i[i])

            rmse = np.sqrt(total_loss / len(shuffled))
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1:2d} | Train RMSE: {rmse:.4f}")

        return self

    def predict(self, user_id, movie_id, timestamp):
        u = self.user2idx.get(user_id)
        i = self.movie2idx.get(movie_id)
        if u is None or i is None:
            return self.mu

        t_norm  = (timestamp - self.t_min) / (self.t_max - self.t_min + 1)
        t_u     = self.t_u_mean.get(user_id, 0.5)
        dev     = self._dev(t_norm, t_u)
        bin_idx = self._get_bin(t_norm)

        b_u_t = self.b_u[u] + self.alpha_u[u] * dev
        b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]
        p_u_t = self.p_u[u] + self.alpha_p[u] * dev

        r_hat = self.mu + b_u_t + b_i_t + np.dot(p_u_t, self.q_i[i])
        return float(np.clip(r_hat, 1, 5))

    def test(self, test_df):
        preds = []
        for _, row in test_df.iterrows():
            est = self.predict(row['userId'], row['movieId'], row['timestamp'])
            preds.append(Prediction(row['userId'], row['movieId'],
                                    row['rating'], est, None))
        return preds

# 학습 실행
time_svdpp2 = TimeSVDpp_v2(n_factors=60, n_epochs=20,
                             lr=0.001, reg=0.02, n_bins=30, beta=0.4)
time_svdpp2.fit(train_df)

print("\nTest 평가 중...")
preds_timesvd2  = time_svdpp2.test(test_df)
rmse_timesvd2   = accuracy.rmse(preds_timesvd2, verbose=False)
mae_timesvd2    = accuracy.mae(preds_timesvd2,  verbose=False)

print(f"\n=== TimeSVD++ v2 결과 ===")
print(f"RMSE: {rmse_timesvd2:.4f}")
print(f"MAE:  {mae_timesvd2:.4f}")
print(f"\nTemporalSVD (A): 0.9380")
print(f"TimeSVD++ v2:    {rmse_timesvd2:.4f}")
print(f"개선폭:           {0.9380 - rmse_timesvd2:.4f}")

TimeSVD++ v2 학습 시작
lr=0.001, n_factors=60, n_bins=30

Epoch  5 | Train RMSE: 0.9331
Epoch 10 | Train RMSE: 0.9096
Epoch 15 | Train RMSE: 0.8999
Epoch 20 | Train RMSE: 0.8942

Test 평가 중...

=== TimeSVD++ v2 결과 ===
RMSE: 1.0177
MAE:  0.8232

TemporalSVD (A): 0.9380
TimeSVD++ v2:    1.0177
개선폭:           -0.0797


In [22]:
class TimeSVDpp_v3:
    """
    단순화된 TimeSVD++
    - 유저/영화 편향만 시간 의존적으로 변경
    - 잠재 요인은 시간 고정 (과적합 방지)
    - 강한 정규화 적용
    """
    def __init__(self, n_factors=60, n_epochs=20, lr=0.002,
                 reg=0.05, n_bins=20, beta=0.4, random_state=42):
        self.n_factors    = n_factors
        self.n_epochs     = n_epochs
        self.lr           = lr
        self.reg          = reg
        self.n_bins       = n_bins
        self.beta         = beta
        self.random_state = random_state

    def _get_bin(self, t_norm):
        return min(int(t_norm * self.n_bins), self.n_bins - 1)

    def _dev(self, t_norm, t_u_mean):
        diff = t_norm - t_u_mean
        return np.sign(diff) * (abs(diff) ** self.beta)

    def fit(self, train_df):
        np.random.seed(self.random_state)

        self.t_min = train_df['timestamp'].min()
        self.t_max = train_df['timestamp'].max()
        t_range    = self.t_max - self.t_min + 1

        train_df = train_df.copy()
        train_df['t_norm'] = (train_df['timestamp'] - self.t_min) / t_range

        users  = train_df['userId'].unique()
        movies = train_df['movieId'].unique()
        self.user2idx  = {u: i for i, u in enumerate(users)}
        self.movie2idx = {m: i for i, m in enumerate(movies)}

        n_users  = len(users)
        n_movies = len(movies)

        self.mu      = train_df['rating'].mean()
        self.b_u     = np.zeros(n_users)
        self.b_i     = np.zeros(n_movies)
        self.alpha_u = np.zeros(n_users)
        self.b_i_t   = np.zeros((n_movies, self.n_bins))
        self.p_u     = np.random.normal(0, 0.01, (n_users,  self.n_factors))
        self.q_i     = np.random.normal(0, 0.01, (n_movies, self.n_factors))

        self.t_u_mean = train_df.groupby('userId')['t_norm'].mean()

        print(f"TimeSVD++ v3 학습 시작")
        print(f"lr={self.lr}, reg={self.reg}, n_factors={self.n_factors}\n")

        best_train_rmse = float('inf')

        for epoch in range(self.n_epochs):
            total_loss = 0
            shuffled   = train_df.sample(frac=1, random_state=epoch)

            for _, row in shuffled.iterrows():
                u = self.user2idx.get(row['userId'])
                i = self.movie2idx.get(row['movieId'])
                if u is None or i is None:
                    continue

                t_norm  = row['t_norm']
                r       = row['rating']
                t_u     = self.t_u_mean.get(row['userId'], 0.5)
                dev     = self._dev(t_norm, t_u)
                bin_idx = self._get_bin(t_norm)

                # 예측 (잠재 요인은 시간 고정)
                b_u_t = self.b_u[u] + self.alpha_u[u] * dev
                b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]

                r_hat = self.mu + b_u_t + b_i_t + np.dot(self.p_u[u], self.q_i[i])
                r_hat = np.clip(r_hat, 1, 5)
                err   = np.clip(r - r_hat, -5, 5)
                total_loss += err ** 2

                self.b_u[u]           += self.lr * (err - self.reg * self.b_u[u])
                self.alpha_u[u]       += self.lr * (err * dev - self.reg * self.alpha_u[u])
                self.b_i[i]           += self.lr * (err - self.reg * self.b_i[i])
                self.b_i_t[i,bin_idx] += self.lr * (err - self.reg * self.b_i_t[i,bin_idx])
                self.p_u[u]           += self.lr * (err * self.q_i[i] - self.reg * self.p_u[u])
                self.q_i[i]           += self.lr * (err * self.p_u[u] - self.reg * self.q_i[i])

            rmse = np.sqrt(total_loss / len(shuffled))
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1:2d} | Train RMSE: {rmse:.4f}")

        return self

    def predict(self, user_id, movie_id, timestamp):
        u = self.user2idx.get(user_id)
        i = self.movie2idx.get(movie_id)
        if u is None or i is None:
            return self.mu

        t_norm  = (timestamp - self.t_min) / (self.t_max - self.t_min + 1)
        t_u     = self.t_u_mean.get(user_id, 0.5)
        dev     = self._dev(t_norm, t_u)
        bin_idx = self._get_bin(t_norm)

        b_u_t = self.b_u[u] + self.alpha_u[u] * dev
        b_i_t = self.b_i[i] + self.b_i_t[i, bin_idx]
        r_hat = self.mu + b_u_t + b_i_t + np.dot(self.p_u[u], self.q_i[i])
        return float(np.clip(r_hat, 1, 5))

    def test(self, test_df):
        preds = []
        for _, row in test_df.iterrows():
            est = self.predict(row['userId'], row['movieId'], row['timestamp'])
            preds.append(Prediction(row['userId'], row['movieId'],
                                    row['rating'], est, None))
        return preds

# 학습
time_svdpp3 = TimeSVDpp_v3(n_factors=60, n_epochs=20,
                             lr=0.002, reg=0.05, n_bins=20, beta=0.4)
time_svdpp3.fit(train_df)

print("\nTest 평가 중...")
preds_timesvd3 = time_svdpp3.test(test_df)
rmse_timesvd3  = accuracy.rmse(preds_timesvd3, verbose=False)
mae_timesvd3   = accuracy.mae(preds_timesvd3,  verbose=False)

print(f"\n=== TimeSVD++ v3 결과 ===")
print(f"RMSE: {rmse_timesvd3:.4f}")
print(f"MAE:  {mae_timesvd3:.4f}")
print(f"\nTemporalSVD (A): 0.9380")
print(f"TimeSVD++ v3:    {rmse_timesvd3:.4f}")
print(f"개선폭:           {0.9380 - rmse_timesvd3:.4f}")

TimeSVD++ v3 학습 시작
lr=0.002, reg=0.05, n_factors=60

Epoch  5 | Train RMSE: 0.9136
Epoch 10 | Train RMSE: 0.8980
Epoch 15 | Train RMSE: 0.8917
Epoch 20 | Train RMSE: 0.8880

Test 평가 중...

=== TimeSVD++ v3 결과 ===
RMSE: 1.0173
MAE:  0.8224

TemporalSVD (A): 0.9380
TimeSVD++ v3:    1.0173
개선폭:           -0.0793
